# 01 – Auditoria dos dados

Dois caminhos: (A) planilha já agregada por mês (formato do LAPAE: uma linha por unidade ou segmento, uma coluna por mês) ou (B) registros brutos, um por exame. Sem nenhum dos dois em `dados/bruto/`, roda em **modo demonstração** (dados sintéticos); nenhum número desse modo é resultado do projeto.

In [ ]:
from pathlib import Path
import sys
RAIZ = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
sys.path.insert(0, str(RAIZ))
import pandas as pd
from src import config
from src.dados import carregar_bruto, carregar_mensal_agregado, limpar, agregar_mensal, validar_totais
from src.demo import gerar_registros_sinteticos, ROTULO_DEMO
pd.set_option("display.width", 140)
agregado = carregar_mensal_agregado()
print("Caminho A (planilha agregada por mês):", agregado is not None)

## Caminho A: planilha agregada
O código só lê `dados/bruto/`; nunca grava nele. Ausência de registro não é convertida em zero.

In [ ]:
if agregado is not None:
    print("Arquivo:", agregado.attrs["arquivo_origem"])
    print("Linhas (unidade x mês):", len(agregado), "| unidades:", agregado[config.COL_UNIDADE].nunique(), "| meses:", agregado[config.COL_MES].nunique())
    print("Período:", f"{agregado[config.COL_MES].min():%Y-%m}", "a", f"{agregado[config.COL_MES].max():%Y-%m}")
    print("Valores ausentes:", int(agregado[config.COL_ALVO].isna().sum()), "| negativos:", int((agregado[config.COL_ALVO] < 0).sum()))
    # conferência: soma da base longa == soma da planilha original
    soma_longa = float(agregado[config.COL_ALVO].sum())
    assert soma_longa == agregado.attrs["total_planilha"], (soma_longa, agregado.attrs["total_planilha"])
    print("Total conferido com a planilha:", int(soma_longa))
    display(agregado.pivot(index=config.COL_UNIDADE, columns=config.COL_MES, values=config.COL_ALVO).rename(columns=lambda c: f"{c:%Y-%m}"))
    display(agregado.groupby(config.COL_MES)[config.COL_ALVO].sum().rename("total do mês").to_frame())

In [ ]:
if agregado is not None:
    agregado.to_csv(config.PROCESSADO / "base_mensal.csv", index=False)
    print("Gravado: dados/processado/base_mensal.csv")

## Caminho B: registros brutos (ou demonstração)

In [ ]:
bruto = None
if agregado is None:
    bruto = carregar_bruto()
    MODO_DEMO = bruto is None
    if MODO_DEMO:
        bruto = gerar_registros_sinteticos()
        print(ROTULO_DEMO)
    print("Dimensões:", bruto.shape)
    print("Unidades (grafias brutas):", bruto[config.COL_UNIDADE].nunique())
    print("Datas ausentes:", int(pd.to_datetime(bruto[config.COL_DATA], errors="coerce").isna().sum()))
    if config.COL_ID in bruto.columns:
        print("Registros com id repetido:", int(bruto[config.COL_ID].duplicated().sum()))

In [ ]:
if bruto is not None:
    limpo, log = limpar(bruto)
    display(log)
    mensal = agregar_mensal(limpo)
    resultado = validar_totais(limpo, mensal)
    print(resultado)
    assert resultado["totais_conferem"], "Total de registros limpos difere da soma mensal"
    nome = "base_mensal_DEMO.csv" if MODO_DEMO else "base_mensal.csv"
    mensal.to_csv(config.PROCESSADO / nome, index=False)
    print("Gravado: dados/processado/" + nome)